# Optimization, Monitoring & Security Hardening · Edge-Computing Notebook

In the Benchmarking lab you measured how your edge system performs. This lab is about acting on those measurements: making inference **faster and lighter**, **monitoring** services so problems are caught early, and **hardening** the system so a shared, physically exposed edge device stays safe.

```text
Measure (benchmark)  ->  Optimize (faster / smaller)
                     ->  Monitor (catch failures)
                     ->  Harden (reduce attack surface)
```

You will:

- speed up YOLO inference with **FP16** and a smaller **input size**, and prove it with data
- right-size a container with **CPU and memory limits** and verify them through cgroups
- inspect **SSH and account security** on the shared DGX Spark
- run a container the **hardened** way: non-root, read-only filesystem, no capabilities
- see why baking **secrets** into images is dangerous · and inject them the right way
- add **health checks and auto-restart**, then build a **threshold alerter**

The main idea is edge devices are constrained, always-on, and often physically reachable by strangers. Good edge engineering means squeezing more out of limited hardware **and** assuming the device could be tampered with.

## How this notebook works · cells and a Jupyter terminal

Most steps in this lab run once and return (an inference sweep, a `docker run`, an inspection command). Those are ordinary **notebook cells** · just run them here.

One step runs live and never returns on its own: the threshold alerter in Part 8. A notebook cell cannot do that · it would hang the kernel. For it, this notebook **writes a small script into your lab folder and asks you to run it in a Jupyter terminal.**

Two labels are used throughout:

- **[Notebook cell]** · run the cell right here.
- **[Terminal]** · open a Jupyter terminal (**File ▸ New ▸ Terminal**, or the Terminal tile in the Launcher) and run the given command there.

### One config, everywhere

Terminal shells do not share the notebook kernel's variables. So the setup cell writes `~/optimizeSecureLab/labEnv.sh` with your `USER`, `PORT`, and `DEVICE_ID`, and every helper script starts with `source ~/optimizeSecureLab/labEnv.sh` · the notebook and your terminals always agree with no manual step.

**Requirements:** a Python 3 kernel on the DGX Spark, Docker available to your user, and the Edge AI lab's venv (`~/edgeAILab/venv`) for the inference cells in Part 2.

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

---
## Part 0 · Before You Begin

You are working on a shared **DGX Spark** edge device · and this notebook runs there too. Your JupyterLab session lives on the DGX Spark, so **every cell in this exercise runs on the DGX Spark**, not on your own computer.

Because the DGX Spark is shared:

- use your own home folder and keep your files under it
- prefix any shared resource names with your account name using `${USER}` (the cells below already do)
- you can only harden **your own** account and containers · device-wide settings (the SSH daemon, firewall, OS updates) are managed by your instructor. This lab has you **inspect** those and change only what is yours.

Some inference steps reuse the Edge AI environment. The Part 2 cells activate it for you (`source ~/edgeAILab/venv/bin/activate` in their own subshell); if your class uses an inference container instead, run the Part 2 commands inside that container the way your instructor showed you.

---
## Part 1 · Set Up Your Working Folder

**[Notebook cell]** The cell below creates `~/optimizeSecureLab`, derives your unique `PORT` from the digits in your username (base 5000, so `student07` gets `5007` · used by the health-check demo in Part 7), sets `DEVICE_ID=${USER}-thor`, and writes `labEnv.sh` so terminal scripts agree with the notebook. To force a specific port, pass `portOverrides={"PORT": <number>}`.

In [ ]:
import os

# PORT is derived from the digits in your username (student07 -> 5007). If your
# instructor assigned you a specific port, add portOverrides={"PORT": <number>}.
labConfig = setupLab(
    "optimizeSecureLab",
    ports={"PORT": 5000},
    extraEnv={"DEVICE_ID": f"{os.environ.get('USER', 'student01')}-thor"},
)

### Preflight · check your environment

Run this once at the start. It confirms Docker is reachable and that the Edge AI inference environment from the earlier lab is available (Part 2 needs it). Fix any failing check before continuing · the last check (the benchmark analyzer) only matters for the compare cell in Part 2.

In [ ]:
import os

preflight(
    [
        check("docker daemon reachable", dockerDaemonUp(),
              hint="Docker must be running and your account must have an instructor-provisioned rootless Podman socket. "
                   "Try <code>docker info</code> in a Terminal; if it fails, ask your instructor.",
              link="https://docs.docker.com/engine/daemon/troubleshoot/",
              linkText="Docker daemon troubleshooting"),
        check("Edge AI venv present (~/edgeAILab/venv)",
              fileExists("~/edgeAILab/venv/bin/activate"),
              hint="Part 2 reuses the Edge AI lab environment. Re-create it from the YOLO lab, "
                   "or use the inference container your instructor provided.",
              link="https://docs.ultralytics.com/quickstart/",
              linkText="Ultralytics quickstart"),
        check("ultralytics importable in that venv",
              commandSucceeds("bash -c 'source ~/edgeAILab/venv/bin/activate && python3 -c \"import ultralytics\"'",
                              timeoutSeconds=120),
              hint="Activate the venv in a Terminal and run <code>pip install ultralytics</code> "
                   "(from the YOLO lab), then re-run this preflight.",
              link="https://docs.ultralytics.com/quickstart/",
              linkText="Ultralytics quickstart"),
        check("benchmark analyzer available (~/benchmarkLab/analyze_latency.py)",
              fileExists("~/benchmarkLab/analyze_latency.py"),
              hint="The compare cell in Part 2 reuses the analyzer you wrote in the Benchmarking lab. "
                   "Finish that lab first, or skip the compare cell.",
              link="https://jsonlines.org/",
              linkText="JSON Lines format"),
    ],
    infoRows=[("your USER", os.environ.get("USER", "?")),
              ("your PORT", os.environ.get("PORT", "?")),
              ("your DEVICE_ID", os.environ.get("DEVICE_ID", "?"))],
)

In [ ]:
!mkdir -p ~/optimizeSecureLab/data
%cd ~/optimizeSecureLab

---
## Part 2 · Optimize by Choosing the Right Configuration

The cheapest optimization is the one you already proved in benchmarking: pick the smallest model and input size that is still accurate enough. Make that improvement measurable.

**[Notebook cell]** Create `optimizedInfer.py`, a latency logger with two new knobs · **input size** (`IMGSZ`) and **half precision** (`HALF`, FP16 on the GPU):

In [ ]:
%%writefile optimizedInfer.py
import os
import json
import time
from datetime import datetime, timezone
from ultralytics import YOLO
from ultralytics.utils import ASSETS

modelName = os.environ.get("MODEL", "yolov8n.pt")
deviceName = os.environ.get("DEVICE", "0")
deviceID = os.environ.get("DEVICE_ID", "thor")
imagePath = os.environ.get("IMAGE", str(ASSETS / "bus.jpg"))
imageSize = int(os.environ.get("IMGSZ", "640"))
useHalf = os.environ.get("HALF", "false").lower() == "true"
runCount = int(os.environ.get("RUNS", "100"))
warmupCount = int(os.environ.get("WARMUP", "5"))
logFile = os.environ.get("LOG_FILE", "data/opt.jsonl")

model = YOLO(modelName)

for _ in range(warmupCount):
    model(imagePath, device=deviceName, imgsz=imageSize, half=useHalf, verbose=False)

os.makedirs(os.path.dirname(logFile) or ".", exist_ok=True)
print(f"{modelName} imgsz={imageSize} half={useHalf} -> {logFile}", flush=True)

with open(logFile, "a") as out:
    for runIndex in range(runCount):
        startTime = time.perf_counter()
        results = model(imagePath, device=deviceName, imgsz=imageSize, half=useHalf, verbose=False)
        endTime = time.perf_counter()
        firstResult = results[0]
        record = {
            "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "device_id": deviceID,
            "model": os.path.basename(str(modelName)),
            "imgsz": imageSize,
            "half": useHalf,
            "compute": "gpu" if deviceName != "cpu" else "cpu",
            "latency_ms": {
                "preprocess": round(firstResult.speed.get("preprocess", 0.0), 2),
                "inference": round(firstResult.speed.get("inference", 0.0), 2),
                "postprocess": round(firstResult.speed.get("postprocess", 0.0), 2),
                "end_to_end": round((endTime - startTime) * 1000, 2),
            },
            "num_detections": len(firstResult.boxes),
        }
        out.write(json.dumps(record) + "\n")
        out.flush()
print("done", flush=True)

In [ ]:
# Preview the latency logger we just wrote.
showFile("~/optimizeSecureLab/optimizedInfer.py", language="python")

**[Notebook cell]** Run a baseline, then two optimizations (50 runs each is plenty to see the difference; the full sweep methodology was covered in the Benchmarking lab). Each line activates the Edge AI venv in its own subshell, so nothing sticks to the notebook. Expect each run to take a minute or so:

In [ ]:
!bash -c 'source ~/edgeAILab/venv/bin/activate && cd ~/optimizeSecureLab && DEVICE=0 IMGSZ=640 HALF=false RUNS=50 LOG_FILE=data/baseline.jsonl   python3 optimizedInfer.py'
!bash -c 'source ~/edgeAILab/venv/bin/activate && cd ~/optimizeSecureLab && DEVICE=0 IMGSZ=640 HALF=true  RUNS=50 LOG_FILE=data/half.jsonl       python3 optimizedInfer.py'
!bash -c 'source ~/edgeAILab/venv/bin/activate && cd ~/optimizeSecureLab && DEVICE=0 IMGSZ=320 HALF=true  RUNS=50 LOG_FILE=data/small_half.jsonl python3 optimizedInfer.py'

**[Notebook cell]** Compare them with the analyzer from the Benchmarking lab:

In [ ]:
!bash -c 'source ~/edgeAILab/venv/bin/activate && cd ~/optimizeSecureLab && python3 ~/benchmarkLab/analyze_latency.py data/baseline.jsonl data/half.jsonl data/small_half.jsonl'

**You should see:** three blocks; the `half` block should show lower latency than `baseline`, and `small_half` lower still · while `num_detections` stays close, not collapsing to zero.

> **If it doesn't work:** `ModuleNotFoundError: ultralytics` means the inference environment is not usable · the cells above `source ~/edgeAILab/venv/bin/activate` for you, so re-check the venv rows in the preflight. A half-precision/CUDA error means you set `HALF=true` with `DEVICE=cpu`; FP16 is GPU-only, so keep `HALF=true` paired with `DEVICE=0`.

The trade-off: a smaller `imgsz` can miss small or distant objects. **Optimization is only valid if you re-measure accuracy, not just speed.**

> **Advanced (instructor-guided):** the biggest Jetson speedups come from exporting the model to a TensorRT engine (`model.export(format="engine", half=True)`). This is version-sensitive and can take several minutes, so only do it with the export settings your instructor provides.

### Checkpoint · Part 2

In [ ]:
import os

checkpoint(
    "Part 2 - optimized inference measured",
    [
        check("optimizedInfer.py exists with the IMGSZ knob",
              fileContains("~/optimizeSecureLab/optimizedInfer.py", "IMGSZ"),
              hint="Re-run the %%writefile optimizedInfer.py cell in Part 2.",
              link="https://docs.ultralytics.com/modes/predict/",
              linkText="Ultralytics predict docs"),
        check("baseline measured (FP32, imgsz 640)",
              jsonLinesValid("~/optimizeSecureLab/data/baseline.jsonl",
                             requiredKeys=["latency_ms", "imgsz", "device_id"], minRecords=10),
              hint="Re-run the first line of the three-run cell in Part 2 (baseline.jsonl).",
              link="https://jsonlines.org/", linkText="JSON Lines format"),
        check("FP16 measured (HALF=true)",
              jsonLinesValid("~/optimizeSecureLab/data/half.jsonl",
                             requiredKeys=["latency_ms", "half"], minRecords=10),
              hint="Re-run the second line of the three-run cell in Part 2 (half.jsonl).",
              link="https://docs.ultralytics.com/modes/predict/#inference-arguments",
              linkText="Inference arguments (half, imgsz)"),
        check("small + FP16 measured (IMGSZ=320)",
              jsonLinesValid("~/optimizeSecureLab/data/small_half.jsonl",
                             requiredKeys=["latency_ms", "imgsz"], minRecords=10),
              hint="Re-run the third line of the three-run cell in Part 2 (small_half.jsonl).",
              link="https://docs.ultralytics.com/modes/predict/#inference-arguments",
              linkText="Inference arguments (half, imgsz)"),
    ],
    successNote="Three measured configurations - the measure -> optimize loop is closed.",
    docLink="https://docs.ultralytics.com/guides/nvidia-jetson/",
    docLinkText="Ultralytics on NVIDIA Jetson",
)

---
## Part 3 · Right-Size the Container

An optimized model still wastes resources if its container is allowed to grab everything on a shared device. Constrain it to what it actually needs, and prove it still runs.

**[Notebook cell]** Start a container with explicit CPU and memory limits, detached (the `rm -f` first makes the cell safe to re-run):

In [ ]:
!docker rm -f ${USER}-limitedTest 2>/dev/null || true
!docker run -d \
  --memory=1g \
  --cpus=2 \
  --name ${USER}-limitedTest \
  alpine sh -c "sleep 600"

**[Notebook cell]** Verify the limits through Docker and through the kernel's cgroup files:

In [ ]:
!docker stats --no-stream ${USER}-limitedTest
!docker exec ${USER}-limitedTest cat /sys/fs/cgroup/memory.max
!docker exec ${USER}-limitedTest cat /sys/fs/cgroup/cpu.max

The `MEM LIMIT` column from `docker stats` shows about 1 GiB, `memory.max` shows `1073741824` (1 GB in bytes), and `cpu.max` shows the CPU quota the container is allowed.

> Note: `nproc` and `free -h` **inside** the container would still report the whole host. The limits are enforced by Linux cgroups, not by hiding hardware from the process · which is why you check the cgroup files, not `free`. (If `memory.max` is missing, this device uses the older cgroup v1; the equivalent path is `/sys/fs/cgroup/memory/memory.limit_in_bytes`.)

### Checkpoint · Part 3

Run this **before** the removal cell below · it inspects the live container.

In [ ]:
import os
userName = os.environ.get("USER", "student01")

checkpoint(
    "Part 3 - container right-sized",
    [
        check("limited test container is running",
              containerRunning(f"{userName}-limitedTest"),
              hint="Re-run the docker run cell at the top of Part 3 (the container sleeps 600s "
                   "and then exits - just start it again).",
              link="https://docs.docker.com/engine/containers/resource_constraints/",
              linkText="Docker resource constraints"),
        check("memory limit is 1 GiB",
              outputContains(["docker", "inspect", "--format", "{{.HostConfig.Memory}}",
                              f"{userName}-limitedTest"], "1073741824"),
              hint="The run cell must include <code>--memory=1g</code>. Re-run it, then this checkpoint.",
              link="https://docs.docker.com/engine/containers/resource_constraints/#memory",
              linkText="Memory limits"),
        check("CPU limit is 2 CPUs",
              outputContains(["docker", "inspect", "--format", "{{.HostConfig.NanoCpus}}",
                              f"{userName}-limitedTest"], "2000000000"),
              hint="The run cell must include <code>--cpus=2</code>. Re-run it, then this checkpoint.",
              link="https://docs.docker.com/engine/containers/resource_constraints/#cpu",
              linkText="CPU limits"),
    ],
    successNote="Limits verified through Docker and enforced by Linux cgroups.",
    docLink="https://docs.kernel.org/admin-guide/cgroup-v2.html",
    docLinkText="Linux cgroup v2 documentation",
)

**[Notebook cell]** Remove the test container:

In [ ]:
!docker rm -f ${USER}-limitedTest 2>/dev/null || true

On a shared edge node, every container should declare limits so one workload cannot starve the others. In Compose this is the `deploy.resources.limits` block or the `mem_limit` / `cpus` keys.

Also keep images small: a smaller base image means faster pulls, less disk, and fewer installed packages that could contain vulnerabilities:

```text
python:3.11        -> large, full toolchain
python:3.11-slim   -> much smaller (what the earlier labs used)
```

This is both an optimization and a security improvement: less software is less attack surface.

---
## Part 4 · Inspect SSH and Account Security

You cannot change the DGX Spark's SSH daemon (the instructor manages it), but you should know how to **inspect** the security of your own access.

**[Notebook cell]** Check your SSH key and folder permissions · too-open permissions are a common, real problem:

In [ ]:
!ls -la ~/.ssh

Recommended permissions:

```text
~/.ssh                 -> 700
~/.ssh/authorized_keys -> 600
private keys           -> 600
```

**[Notebook cell]** Fix yours if needed:

In [ ]:
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/authorized_keys 2>/dev/null || true

**[Notebook cell]** See what your account is allowed to do with elevated privileges, and who else is logged in to the shared device right now. (`sudo -n` is the non-interactive flag · without it, a password prompt would hang the notebook cell.)

In [ ]:
!sudo -n -l 2>/dev/null || echo "no sudo access (expected for a student account)"
!who

Key principles you are observing:

```text
Key-based auth  -> stronger than passwords; the device should disable password login
Least privilege -> a student account should not be able to touch other accounts or the OS
Shared device   -> assume others are on it; never store secrets in world-readable files
```

### Checkpoint · Part 4

In [ ]:
import os

checkpoint(
    "Part 4 - SSH and account security inspected",
    [
        check("~/.ssh directory exists", dirExists("~/.ssh"),
              hint="No ~/.ssh yet? Revisit the SSH, Linux, Git, GitHub lab where your key access "
                   "was set up, or create it with <code>mkdir -m 700 ~/.ssh</code>.",
              link="https://www.openssh.com/manual.html", linkText="OpenSSH manual pages"),
        check("~/.ssh permissions are 700",
              outputContains("stat -c %a ~/.ssh", "700"),
              hint="Re-run the chmod cell in Part 4 (<code>chmod 700 ~/.ssh</code>).",
              link="https://man.openbsd.org/sshd.8#FILES", linkText="sshd files and permissions"),
        check("authorized_keys is 600 (or absent)",
              commandSucceeds("bash -c '[ ! -f ~/.ssh/authorized_keys ] || [ \"$(stat -c %a ~/.ssh/authorized_keys)\" = \"600\" ]'"),
              hint="Re-run the chmod cell in Part 4 (<code>chmod 600 ~/.ssh/authorized_keys</code>).",
              link="https://man.openbsd.org/sshd.8#FILES", linkText="sshd files and permissions"),
    ],
    successNote="Your own access is locked down - least privilege in practice.",
    docLink="https://www.openssh.com/manual.html",
    docLinkText="OpenSSH manual pages",
)

---
## Part 5 · Harden Your Containers

This is where you have full control. Run a container the **insecure** default way, then the **hardened** way, and compare.

**[Notebook cell]** Insecure default (runs as root, full capabilities, writable filesystem):

In [ ]:
!docker run --rm alpine sh -c "whoami; id"

It reports `root`. A process escaping that container would do so as root.

**[Notebook cell]** Now run it hardened (the output is also saved to `data/hardenedRun.txt` so the checkpoint can verify it):

In [ ]:
!mkdir -p ~/optimizeSecureLab/data
!docker run --rm \
  --user 1000:1000 \
  --read-only \
  --tmpfs /tmp \
  --cap-drop ALL \
  --security-opt no-new-privileges \
  --memory=256m \
  alpine sh -c "whoami; id; touch /should-fail 2>&1 || echo 'root filesystem is read-only (good)'" | tee ~/optimizeSecureLab/data/hardenedRun.txt

What each flag does:

```text
--user 1000:1000          -> run as a non-root user, not root
--read-only               -> the container filesystem cannot be modified
--tmpfs /tmp              -> give it only a small writable scratch area
--cap-drop ALL            -> remove all Linux capabilities it does not need
--security-opt no-new-privileges -> the process can never gain more privileges
--memory=256m             -> cap memory so it cannot exhaust the device
```

Real edge services should run hardened like this by default. The same options exist in Compose (`user:`, `read_only:`, `cap_drop:`, `security_opt:`, `tmpfs:`).

### Checkpoint · Part 5

In [ ]:
import os

checkpoint(
    "Part 5 - container hardened",
    [
        check("hardened run happened as uid 1000 (non-root)",
              fileContains("~/optimizeSecureLab/data/hardenedRun.txt", "uid=1000"),
              hint="Re-run the hardened docker run cell in Part 5 - it must include "
                   "<code>--user 1000:1000</code> and the <code>| tee ...</code> at the end.",
              link="https://docs.docker.com/engine/security/",
              linkText="Docker security overview"),
        check("root filesystem was read-only",
              fileContains("~/optimizeSecureLab/data/hardenedRun.txt", "read-only"),
              hint="Re-run the hardened docker run cell in Part 5 - it must include "
                   "<code>--read-only</code> (the touch /should-fail test proves it).",
              link="https://docs.docker.com/reference/cli/docker/container/run/#read-only",
              linkText="docker run --read-only"),
        check("hardened flags still work live (quick re-run)",
              outputContains("docker run --rm --user 1000:1000 --cap-drop ALL "
                             "--security-opt no-new-privileges alpine id", "uid=1000",
                             timeoutSeconds=90),
              hint="Docker could not run the hardened alpine container - check the daemon and "
                   "re-run the Part 5 cells.",
              link="https://cheatsheetseries.owasp.org/cheatsheets/Docker_Security_Cheat_Sheet.html",
              linkText="OWASP Docker security cheat sheet"),
    ],
    successNote="Non-root, read-only, no capabilities, no privilege escalation - this is how edge services should ship.",
    docLink="https://cheatsheetseries.owasp.org/cheatsheets/Docker_Security_Cheat_Sheet.html",
    docLinkText="OWASP Docker security cheat sheet",
)

---
## Part 6 · Keep Secrets Out of Images and Git

A very common edge security failure is baking a token or password into an image or committing it to Git. Demonstrate why that is dangerous. (The "secret" here is a harmless placeholder · never do this with a real one.)

**[Notebook cell]** Build a tiny image that bakes in a secret (the wrong way):

In [ ]:
!mkdir -p ~/optimizeSecureLab/secretDemo
%cd ~/optimizeSecureLab/secretDemo

In [ ]:
%%writefile Dockerfile
FROM alpine
ENV API_TOKEN=super-secret-12345
CMD ["sh", "-c", "echo running"]

In [ ]:
# Preview the (deliberately bad) Dockerfile we just wrote.
showFile("~/optimizeSecureLab/secretDemo/Dockerfile", language="docker")

**[Notebook cell]** Build it and then reveal the secret straight out of the image metadata:

> **Naming reminder:** `${USER}-leaky` here is an *image* name, so Docker requires it to be lowercase. Earlier in this lab, `${USER}-limitedTest` was a *container* name, which may use capitals. Same lab, two namespaces, two rules. See Lab 02, Part 13 for why.

In [ ]:
!docker build -t ${USER}-leaky .
!docker inspect ${USER}-leaky | grep API_TOKEN
!docker history --no-trunc ${USER}-leaky | grep API_TOKEN

**You should see:** both `grep` commands print lines containing `super-secret-12345`. That is the lesson: the "secret" baked into the image is readable by anyone who can pull or inspect it; it is not hidden at all.

**[Notebook cell]** Now do it the **right** way · pass the secret at run time, never bake it in. The container only reports *whether* the token is set; it never prints the value:

In [ ]:
!echo "API_TOKEN=super-secret-12345" > secret.env
!chmod 600 secret.env
!docker run --rm --env-file secret.env alpine sh -c 'echo "token is set: ${API_TOKEN:+yes}"'

**[Notebook cell]** And keep the secret file out of Git:

In [ ]:
!printf 'secret.env\n*.env\n' > .gitignore
showFile("~/optimizeSecureLab/secretDemo/.gitignore", language="text", title=".gitignore")

```text
Bad:  secrets in Dockerfile ENV, in image layers, or committed to Git
Good: secrets injected at run time via --env-file or a secrets manager, files chmod 600, .gitignore'd
```

### Checkpoint · Part 6

Run this **before** the cleanup cell below removes the leaky image.

In [ ]:
import os
userName = os.environ.get("USER", "student01")

checkpoint(
    "Part 6 - secrets handled correctly",
    [
        check("leaky Dockerfile demonstrates the anti-pattern",
              fileContains("~/optimizeSecureLab/secretDemo/Dockerfile", "API_TOKEN"),
              hint="Re-run the %%writefile Dockerfile cell in Part 6.",
              link="https://docs.docker.com/build/building/secrets/",
              linkText="Docker build secrets (the right way)"),
        check("the leak is visible in the image metadata",
              outputContains(f"docker history --no-trunc {userName}-leaky", "super-secret-12345"),
              hint="Re-run the build cell in Part 6 (docker build -t <you>-leaky .). If you already "
                   "removed the image with the cleanup cell, that is fine - rebuild it once to re-verify.",
              link="https://docs.docker.com/reference/cli/docker/image/history/",
              linkText="docker history reference"),
        check("secret.env exists with 600 permissions",
              commandSucceeds("bash -c '[ \"$(stat -c %a ~/optimizeSecureLab/secretDemo/secret.env)\" = \"600\" ]'"),
              hint="Re-run the right-way cell in Part 6 (echo ... > secret.env; chmod 600 secret.env).",
              link="https://docs.docker.com/reference/cli/docker/container/run/#env",
              linkText="docker run --env-file"),
        check(".gitignore excludes env files",
              fileContains("~/optimizeSecureLab/secretDemo/.gitignore", "*.env"),
              hint="Re-run the .gitignore cell in Part 6.",
              link="https://git-scm.com/docs/gitignore",
              linkText="gitignore documentation"),
    ],
    successNote="Secrets stay out of images and out of Git - injected at run time only.",
    docLink="https://cheatsheetseries.owasp.org/cheatsheets/Secrets_Management_Cheat_Sheet.html",
    docLinkText="OWASP secrets management cheat sheet",
)

**[Notebook cell]** Clean up the leaky image:

In [ ]:
!docker rmi ${USER}-leaky 2>/dev/null || true

---
## Part 7 · Add Health Checks and Auto-Restart

Monitoring starts with the service telling you whether it is alive. Add a health check and a restart policy so a crashed service is detected and recovered automatically.

**[Notebook cell]** Run a container with a health check and restart policy (your `PORT` from Part 1 is already exported):

> The `nginx:alpine` image is used here because its busybox shell includes `wget` for the health check (the default `nginx` image has neither `wget` nor `curl`).

In [ ]:
!docker rm -f ${USER}-monitored 2>/dev/null || true
!docker run -d \
  --name ${USER}-monitored \
  --restart unless-stopped \
  --health-cmd="wget -q -O- http://localhost:80 || exit 1" \
  --health-interval=10s \
  --health-retries=3 \
  -p $PORT:80 \
  nginx:alpine

**[Notebook cell]** Check the health status:

In [ ]:
import os
userName = os.environ.get("USER", "student01")

dockerPs(namePattern=f"{userName}-monitored")
healthOut, _ = runShell(["docker", "inspect", "--format", "{{.State.Health.Status}}",
                         f"{userName}-monitored"])
showNote(f"health status: <b>{healthOut.strip() or 'unknown'}</b>", kind="info",
         title="docker inspect .State.Health.Status")

**You should see:** the `status` column reads something like `Up 30 seconds (healthy)`; the inspect note prints `healthy`. Right after starting it may briefly show `starting` while the first checks run · wait ~30s and re-run the cell.

**[Notebook cell]** Test recovery · stop NGINX inside the container and watch Docker restart it:

In [ ]:
!docker exec ${USER}-monitored nginx -s stop 2>/dev/null || true
!sleep 8

In [ ]:
import os
userName = os.environ.get("USER", "student01")

dockerPs(namePattern=f"{userName}-monitored")
showNote("Stopping NGINX made the container exit - and the <code>unless-stopped</code> restart "
         "policy brought it right back (see the fresh 'Up ... seconds' status).", kind="ok")

```text
Health check    -> the service reports alive / not alive
Restart policy  -> Docker recovers a crashed service automatically
```

These are the building blocks of edge monitoring, which you will scale to a whole fleet in the Fleet Management lab. Leave `${USER}-monitored` running for the checkpoint below · the Cleanup section at the end removes it.

### Checkpoint · Part 7

In [ ]:
import os
userName = os.environ.get("USER", "student01")

checkpoint(
    "Part 7 - health check and auto-restart",
    [
        check("monitored container is running",
              containerRunning(f"{userName}-monitored"),
              hint="Re-run the docker run cell at the top of Part 7.",
              link="https://docs.docker.com/engine/containers/start-containers-automatically/",
              linkText="Restart policies"),
        check("health check is defined (wget probe)",
              outputContains(["docker", "inspect", "--format", "{{json .Config.Healthcheck}}",
                              f"{userName}-monitored"], "wget"),
              hint="The run cell must include the <code>--health-cmd</code> flag. Re-run it.",
              link="https://docs.docker.com/reference/dockerfile/#healthcheck",
              linkText="HEALTHCHECK reference"),
        check("container reports healthy",
              outputContains(["docker", "inspect", "--format", "{{.State.Health.Status}}",
                              f"{userName}-monitored"], "healthy"),
              hint="Right after starting, the status is 'starting' - wait ~30 seconds and re-run "
                   "this checkpoint. If it reads 'unhealthy', check the container logs.",
              link="https://docs.docker.com/reference/cli/docker/container/inspect/",
              linkText="docker inspect reference"),
        check("restart policy is unless-stopped",
              outputContains(["docker", "inspect", "--format", "{{.HostConfig.RestartPolicy.Name}}",
                              f"{userName}-monitored"], "unless-stopped"),
              hint="The run cell must include <code>--restart unless-stopped</code>. Re-run it.",
              link="https://docs.docker.com/engine/containers/start-containers-automatically/",
              linkText="Restart policies"),
        check("dashboard answers on your PORT",
              portListening(int(os.environ.get("PORT", "5001"))),
              hint="The run cell publishes <code>-p $PORT:80</code>. Confirm PORT in the Part 1 env "
                   "card and that no other service holds it.",
              link="https://docs.docker.com/engine/network/#published-ports",
              linkText="Published ports"),
    ],
    successNote="Detected and recovered automatically - the smallest possible monitoring loop.",
    docLink="https://docs.docker.com/engine/containers/start-containers-automatically/",
    docLinkText="Start containers automatically",
)

---
## Part 8 · Threshold Alerting on Telemetry

Monitoring also means watching the **data** and reacting to bad values. Build a small alerter over the telemetry format you standardized earlier.

**[Notebook cell]** Write the alerter and its runner script:

In [ ]:
%cd ~/optimizeSecureLab

In [ ]:
%%writefile alerter.py
import json
import random
import time
from datetime import datetime, timezone

tempLimit = 38.0  # alert threshold in Celsius

while True:
    reading = {
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "temperature": round(random.uniform(22, 42), 2),
    }
    status = "ALERT" if reading["temperature"] >= tempLimit else "ok"
    print(f"{reading['timestamp']}  temp={reading['temperature']}  {status}", flush=True)
    if status == "ALERT":
        print(f"  -> temperature {reading['temperature']} exceeded {tempLimit}", flush=True)
    time.sleep(1)

In [ ]:
# Preview the alerter we just wrote.
showFile("~/optimizeSecureLab/alerter.py", language="python")

In [ ]:
%%writefile runAlerter.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/optimizeSecureLab/labEnv.sh
cd ~/optimizeSecureLab
mkdir -p data
echo "Threshold alerter starting. Output also appends to data/alerts.log. Press Ctrl-C to stop."
python3 alerter.py | tee -a data/alerts.log

In [ ]:
!chmod +x runAlerter.sh

**[Terminal]** The alerter loops forever, so run it in a Jupyter terminal. Let it run for ~30 seconds · long enough to catch a few readings over the 38.0 °C threshold · then stop it with **Ctrl-C**:

```bash
~/optimizeSecureLab/runAlerter.sh
```

When a reading crosses the threshold, the monitor flags it (and `tee` appends a copy to `data/alerts.log`, which the checkpoint below verifies). In a real system this would page an operator, write to a log, or trigger an automatic action. Notice this is **edge-local** monitoring: the device decides something is wrong without waiting on the cloud.

### Checkpoint · Part 8

In [ ]:
checkpoint(
    "Part 8 - threshold alerting",
    [
        check("alerter.py exists with a threshold",
              fileContains("~/optimizeSecureLab/alerter.py", "tempLimit"),
              hint="Re-run the %%writefile alerter.py cell in Part 8.",
              link="https://docs.python.org/3/library/time.html#time.sleep",
              linkText="Python time.sleep"),
        check("runAlerter.sh helper exists",
              fileExists("~/optimizeSecureLab/runAlerter.sh"),
              hint="Re-run the %%writefile runAlerter.sh and chmod cells in Part 8.",
              link="https://www.gnu.org/software/bash/manual/bash.html",
              linkText="Bash reference manual"),
        check("alerts.log captured some readings",
              fileNonEmpty("~/optimizeSecureLab/data/alerts.log", minLines=5),
              hint="Run <code>~/optimizeSecureLab/runAlerter.sh</code> in a Terminal for at "
                   "least ~10 seconds, then re-run this checkpoint.",
              link="https://man7.org/linux/man-pages/man1/tee.1.html",
              linkText="tee manual"),
        check("at least one ALERT was raised",
              fileContains("~/optimizeSecureLab/data/alerts.log", "ALERT"),
              hint="Readings are random (22-42 vs a 38.0 threshold), so let the alerter run ~30 "
                   "seconds until you see an ALERT line, then re-run this checkpoint.",
              link="https://docs.python.org/3/library/random.html#random.uniform",
              linkText="random.uniform"),
    ],
    successNote="Edge-local monitoring works: the device flags bad values itself, no cloud needed.",
    docLink="https://docs.docker.com/engine/containers/start-containers-automatically/",
    docLinkText="Pair alerting with auto-restart",
)

---
## Cleanup

**[Notebook cell]** Remove any leftover containers and images you created (safe to run any number of times):

In [ ]:
!docker rm -f ${USER}-monitored ${USER}-limitedTest 2>/dev/null || true
!docker rmi ${USER}-leaky 2>/dev/null || true

**[Notebook cell]** Optional · remove your lab folder (uncomment to run). Only remove your own files, containers, and images. Do not change any device-wide settings.

In [ ]:
# !rm -rf ~/optimizeSecureLab

### Lab scorecard

Run this any time for a live summary of every checkpoint in this kernel session. If a row is incomplete, scroll up, fix that part, re-run its checkpoint, and re-run this cell.

In [ ]:
labSummary("Optimization, Monitoring & Security Hardening")

---
## Connect to the Rest of the Course

You closed the measure-improve loop and added the operational concerns a real deployment needs:

```text
Benchmark (measure)  ->  Optimize (smaller model, FP16, right-sized containers)
                     ->  Monitor (health checks, restart, threshold alerts)
                     ->  Harden (non-root, read-only, dropped caps, secrets out of images)
```

A real project applies this continuously: optimize against the benchmark, monitor in production, and harden because an edge device can be physically accessed. The main idea: getting a system to work is the start; making it fast, observable, and secure on constrained shared hardware is what makes it deployable.

---
## Key Terms

- **FP16 / half precision:** running the model with 16-bit floats instead of 32-bit; on the GPU this is faster and uses less memory, with a small accuracy trade-off.
- **Input size (`imgsz`):** the resolution images are resized to before inference; smaller is faster but can miss small or distant objects.
- **TensorRT engine:** an NVIDIA-optimized, compiled form of a model that runs much faster on Jetson (version-sensitive; instructor-guided).
- **cgroups:** the Linux mechanism that enforces a container's CPU/memory limits; you read `/sys/fs/cgroup/*` to confirm the limits applied.
- **Attack surface:** everything an attacker could target; smaller images and fewer privileges shrink it.
- **Container hardening:** running a container with reduced privileges · non-root user, read-only filesystem, `--cap-drop ALL`, `no-new-privileges`.
- **Least privilege:** giving an account or process only the access it needs and nothing more.
- **Health check / restart policy:** a container's self-reported alive/not-alive status, and the rule that auto-restarts it if it crashes.

---
### One-minute feedback

Your feedback shapes the next version of this lab. Rate it, add anything that was confusing or broken, and click **Submit**. It takes about 30 seconds and goes straight to the instructor.

In [ ]:
feedback("Optimization, Monitoring and Security Hardening")